In [0]:
# CELL 1
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

df = spark.read.table("sarkarimitracatalog.sarkaridatabase.raw_data")
print(f"✅ Loaded {df.count()} rows")
df.printSchema()

✅ Loaded 3399 rows
root
 |-- scheme_name: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- details: string (nullable = true)
 |-- benefits: string (nullable = true)
 |-- eligibility: string (nullable = true)
 |-- application: string (nullable = true)
 |-- documents: string (nullable = true)
 |-- level: string (nullable = true)
 |-- schemeCategory: string (nullable = true)
 |-- Unnamed: 9: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- age_min: double (nullable = true)
 |-- age_max: double (nullable = true)
 |-- income_limit_inr: double (nullable = true)
 |-- gender_eligibility: string (nullable = true)
 |-- category_eligibility: string (nullable = true)
 |-- applicable_states: string (nullable = true)
 |-- occupation_types: string (nullable = true)
 |-- family_status: string (nullable = true)
 |-- disability_required: boolean (nullable = true)
 |-- minimum_education: string (nullable = true)
 |-- extraction_confidence: double (nullable = true)



In [0]:
# CELL 2 — Clean and cast types
df_clean = df \
    .drop("Unnamed: 9") \
    .withColumn("age_min",               F.col("age_min").cast(IntegerType())) \
    .withColumn("age_max",               F.col("age_max").cast(IntegerType())) \
    .withColumn("income_limit_inr",      F.col("income_limit_inr").cast(FloatType())) \
    .withColumn("extraction_confidence", F.col("extraction_confidence").cast(FloatType())) \
    .withColumn("disability_required",   F.col("disability_required").cast("boolean")) \
    .fillna({
        "age_min": 0,
        "age_max": 999,
        "income_limit_inr": 9999999.0,
        "gender_eligibility": "all",
        "category_eligibility": "all",
        "applicable_states": "all",
        "disability_required": False,
        "minimum_education": "none",
        "extraction_confidence": 0.5
    }) \
    .filter(F.col("scheme_name").isNotNull())

print(f"✅ Clean rows: {df_clean.count()}")

✅ Clean rows: 3399


In [0]:
# CELL 3 — Check confidence distribution
display(df_clean.groupBy(
    F.when(F.col("extraction_confidence") >= 0.9, "high")
     .when(F.col("extraction_confidence") >= 0.7, "medium")
     .otherwise("low").alias("confidence_band")
).count())

confidence_band,count
high,2756
medium,468
low,175


In [0]:
# CELL 4 — Write
df_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sarkarimitracatalog.sarkaridatabase.schemes_structured")

print("✅ Written: sarkarimitracatalog.sarkaridatabase.schemes_structured")
display(df_clean.limit(5))

✅ Written: sarkarimitracatalog.sarkaridatabase.schemes_structured


scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags,age_min,age_max,income_limit_inr,gender_eligibility,category_eligibility,applicable_states,occupation_types,family_status,disability_required,minimum_education,extraction_confidence
Full Medical Checkup Assistance Scheme- Gujarat Labour Welfare Board,sfmcas-glwb,"The “Full Medical Checkup Assistance Scheme” is implemented by the Gujarat Labour Welfare Board, Labour, Skill Development & Employment Department, Government of Gujarat. The objective of the scheme is to conduct health check-ups for male Shramyogis above 45 years of age and female Shram Yogis above 35 years of age who are working in the organized sector in the state of Gujarat. Only those workers (Shramyogi) who’s Labour Welfare Fund has been paid regularly by their organization/unit for the last one year will be eligible.",The scheme provides health check-ups for male Shramyogi above 45 years of age and female Shramyogi above 35 years of age working in the organized sector in the state of Gujarat. Note: A fixed amount of ₹1950/- per worker for complete medical check-ups and 25% of this amount is ₹487.50 to be paid by the organization/company.,"Only those workers (Shramyogis) whose Labour Welfare Fund has been regularly paid by their organization/unit for the last one year will be eligible. The male Shram Yogis above 45 years of age and female Shram Yogis above 35 years of age working in the organized sector will be eligible to avail of the benefits under the scheme. The benefit will be given once a year to the labourers. The Gujarat Labour Welfare Board shall not be responsible for any side effects during/after physical examination. After getting in-principle approval from the office here, the medical examination has to be done within 30 days with the mutual agreement of the company and the hospital. Note 01: The company willing to avail the benefits will have to upload the list (Excel Sheet) in the portal. Note 02: Fees paid under this scheme will not be refunded. Note 03: The final decision regarding the scheme will be with the Welfare Commissioner. The jurisdiction will be Ahmedabad.","Application Process for Scheme Benefit: Step 01: The applicant may visit the Sanman Portal: https://sanman.gujarat.gov.in/ ﻿ Step 02: On the home page, under the tab ‘Citizen Login’, click on ‘ Please Register Here ’. Step 03: Enter your Aadhaar Card Number, select user type, and then enter your Labour Welfare Fund Account Number. Step 04: Now, click on ‘Fetch’ & verify the details. Step 05: Enter user details and Password. Step 06: After successful registration, the applicants can login through their User ID and Password. Step 07: Now, select the scheme and read the instructions carefully for the selected scheme. Step 08: Fill out the application form and upload all the relevant documents. Step 09: Agree with the Rules & Regulations and submit the application form. A confirmation email with the application Number will be sent on the registered email ID.",Passport-size Photograph A copy of the identity card issued by the contractor to the worker/labour Aadhaar Card of Labour Labour Welfare Fund Account Number ﻿ Bonafide certificate ﻿ Any other documents as required,State,Health & Wellness,"Shramyogi, Medical, Checkup, Organized Worker, Labour",35,60,9999999.0,female,"[""General""]","[""Gujarat""]","[""salaried""]","[""any""]",false,none,1.0
Funeral Assistance,fa,"The scheme “Funeral Assistance” is implemented by the Maharashtra Building and Other Construction Workers Welfare Board (MBOCWW), Labour Department, Government of Maharashtra. Under this scheme, the Board provides financial assistance for funeral expenses to the nominated heir of a registered worker in the event of their death.","In case of the death of a registered worker, an amount of ₹10,000/- is payable as funeral assistance to the nominated heir.",The applicant should be a nominee/legal heir of the deceased worker. The deceas